# Evaluate PubMedBERT (PMB) on MedQA with full fine-tuning

Files needed:
*   MedQA data (train, valid, test)




Make sure to define the paths to the MedQA data

In [ ]:
path_to_train = "./train.jsonl"
path_to_dev = "./dev.jsonl"
path_to_test = "./test.jsonl"

In [ ]:
! pip3 install transformers datasets torch accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Load base data

One thing to note about the **meta_info** feature :

step 1 ---> basic science

step 2 & 3 ---> clinical knowledge

In [ ]:
import pandas as pd
import os

train_df = pd.read_json(path_to_train, lines=True)
dev_df = pd.read_json(path_to_dev, lines=True)
test_df = pd.read_json(path_to_test, lines=True)

In [ ]:
train_df.info()
dev_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10178 entries, 0 to 10177
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    10178 non-null  object
 1   answer      10178 non-null  object
 2   options     10178 non-null  object
 3   meta_info   10178 non-null  object
 4   answer_idx  10178 non-null  object
dtypes: object(5)
memory usage: 397.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1272 entries, 0 to 1271
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    1272 non-null   object
 1   answer      1272 non-null   object
 2   options     1272 non-null   object
 3   meta_info   1272 non-null   object
 4   answer_idx  1272 non-null   object
dtypes: object(5)
memory usage: 49.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1273 entries, 0 to 1272
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtyp

In [ ]:
train_df.head()

,question,answer,options,meta_info,answer_idx
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin,"{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': '...",step2&3,E
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...,{'A': 'Placing the infant in a supine position...,step2&3,A
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud,{'A': 'Abnormal migration of ventral pancreati...,step1,A
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism,"{'A': 'Thromboembolism', 'B': 'Pulmonary ische...",step1,A
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease,"{'A': 'Factor V Leiden', 'B': 'Hemophilia A', ...",step1,E


# Now let's get serious

In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")

def preprocess_medqa(examples):
    """Tokenize question and choices separately for multiple-choice classification"""
    inputs = {"input_ids": [], "attention_mask": [], "token_type_ids": [], "labels": []}

    for example in tqdm(examples, total=len(examples)):
    #for i in range(len(examples["question"])):  # Process each example individually
        question = example["question"]
        options = example["options"] # Dictionary {'A': 'Ampicillin', 'B': 'Ceftriaxone', ...}
        correct_answer = example["answer_idx"]  # Single letter ('A', 'B', ...)

        # Ensure consistent option order (sort by key)
        option_keys = sorted(options.keys())
        option_values = [options[key] for key in option_keys]  # List of answer choices

        # Convert correct answer letter to index
        if correct_answer in option_keys:
            label = option_keys.index(correct_answer)  # Map the letter to index (0, 1, ...)
        else:
            raise ValueError(f"Unexpected answer key: {correct_answer} in {option_keys}")

        # Tokenize question-answer pairs
        encoding = tokenizer(
            [question] * len(option_values),  # Repeat the question for each choice
            option_values,  # List of choices
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # Append results to inputs dictionary
        inputs["input_ids"].append(encoding["input_ids"].squeeze(0))  # Remove extra batch dimension
        inputs["attention_mask"].append(encoding["attention_mask"].squeeze(0))
        inputs["token_type_ids"].append(encoding.get("token_type_ids", None).squeeze(0) if "token_type_ids" in encoding else None)
        inputs["labels"].append(label)  # Correct answer index

    return inputs

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

In [ ]:
import torch
from datasets import Dataset


# Convert DataFrame to Hugging Face Dataset
hf_train = Dataset.from_pandas(train_df)

# Apply tokenization
train_inputs = preprocess_medqa(hf_train)

hf_valid = Dataset.from_pandas(dev_df)
valid_inputs = preprocess_medqa(hf_valid)

# Let's try converting them back to the Dataset class
train_inputs = Dataset.from_dict(train_inputs)
valid_inputs = Dataset.from_dict(valid_inputs)



100%|██████████| 1272/1272 [00:05<00:00, 229.53it/s]


In [ ]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Verifying if we are working on the GPU
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # GPU name
model.to("cuda")
print(next(model.parameters()).device)  # Should return: cuda:0

True
1
Tesla T4
cuda:0


In [ ]:
import evaluate

# Load metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    return {"accuracy": accuracy["accuracy"]}

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    run_name="PMB_baseline_full",
    output_dir="./",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    fp16=True,  # Enable mixed precision (faster training)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,  # Speed up data loading
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_inputs,
    eval_dataset=valid_inputs,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Pass the metric function
)

<ipython-input-14-76434843a784>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.587300,1.550235,0.297170
2,1.463900,1.524449,0.322327
3,1.308800,1.673540,0.330189
4,0.942300,1.820628,0.328616
5,0.808600,1.981558,0.335692


TrainOutput(global_step=3185, training_loss=1.1814058456540668, metrics={'train_runtime': 2964.0134, 'train_samples_per_second': 17.169, 'train_steps_per_second': 1.075, 'total_flos': 3.34740034659072e+16, 'train_loss': 1.1814058456540668, 'epoch': 5.0})

In [ ]:
trainer.train()

Save model (optional)

In [ ]:
trainer.save_model("models/")
tokenizer.save_pretrained("models/")
print("Model saved")

Model saved


Test on test data

In [ ]:
hf_test = Dataset.from_pandas(test_df)
test_inputs = Dataset.from_dict(preprocess_medqa(hf_test))

results = trainer.evaluate(test_inputs)
with open("./results.csv", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")
print(results)

100%|██████████| 1273/1273 [00:04<00:00, 273.78it/s]


{'eval_loss': 1.524336814880371, 'eval_accuracy': 0.32992930086410055, 'eval_runtime': 23.5308, 'eval_samples_per_second': 54.099, 'eval_steps_per_second': 3.4, 'epoch': 5.0}


# Load an already trained model and evaluate it separately on "step1" questions and "step2"/"step3" questions (optional)

In [ ]:
from transformers import AutoModelForMultipleChoice, AutoTokenizer

path_to_model = "./first_baseline/models"  # Path to saved model and tokenizer

# Load the trained model
model = AutoModelForMultipleChoice.from_pretrained(path_to_model)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(path_to_model)

print("Model and tokenizer loaded successfully!")

Model and tokenizer loaded successfully!


Separate test set by "step" number:

In [ ]:
test_step1 = test_df[test_df["meta_info"] == "step1"]
test_step2_3 = test_df[test_df["meta_info"] == "step2&3"]

test_step1.info()
test_step2_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 679 entries, 0 to 1272
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    679 non-null    object
 1   answer      679 non-null    object
 2   options     679 non-null    object
 3   meta_info   679 non-null    object
 4   answer_idx  679 non-null    object
dtypes: object(5)
memory usage: 31.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 594 entries, 2 to 1271
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    594 non-null    object
 1   answer      594 non-null    object
 2   options     594 non-null    object
 3   meta_info   594 non-null    object
 4   answer_idx  594 non-null    object
dtypes: object(5)
memory usage: 27.8+ KB


Preprocessing of test data

In [ ]:
hf_test_step1 = Dataset.from_pandas(test_step1)
test_step1_inputs = Dataset.from_dict(preprocess_medqa(hf_test_step1))

hf_test_step2_3 = Dataset.from_pandas(test_step2_3)
test_step2_3_inputs = Dataset.from_dict(preprocess_medqa(hf_test_step2_3))

100%|██████████| 594/594 [00:02<00:00, 264.09it/s]


Evaluate model on the two test sets

In [ ]:
import torch

# Move the model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, ele

In [ ]:
from transformers import Trainer, TrainingArguments

# Define evaluation arguments
eval_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=16,   # Adjust batch size for GPU
    dataloader_num_workers=4,        # Speed up evaluation
    fp16=True if torch.cuda.is_available() else False,
    evaluation_strategy="no"
)

trainer_step1 = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=test_step1_inputs,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Add metrics
)

trainer_step2_3 = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=test_step2_3_inputs,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-17-5e2204ec8605>:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_step1 = Trainer(
<ipython-input-17-5e2204ec8605>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_step2_3 = Trainer(


In [ ]:
# Ensure model is on GPU
model.to(device)

# Evaluate on test_step1_inputs
results_step1 = trainer_step1.evaluate()
print("Results for Step 1:", results_step1)

# Evaluate on test_step2_3_inputs
results_step2_3 = trainer_step2_3.evaluate()
print("Results for Step 2-3:", results_step2_3)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: philschoeb (agent_hospital2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Results for Step 1: {'eval_loss': 1.5374491214752197, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.31516936671575846, 'eval_runtime': 11.4344, 'eval_samples_per_second': 59.382, 'eval_steps_per_second': 3.761}


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Results for Step 2-3: {'eval_loss': 1.5093272924423218, 'eval_model_preparation_time': 0.0053, 'eval_accuracy': 0.3468013468013468, 'eval_runtime': 9.943, 'eval_samples_per_second': 59.741, 'eval_steps_per_second': 3.822}
